In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# Define the base path and input files
base_path = r"C:\Users\wanglab\Desktop\Club Like Endings\102725_1\102725_1_shortened"

input_files = [
    Path(base_path) / "mask_lines.csv",
    Path(base_path) / "mask_lines+media_mod.csv",
    Path(base_path) / "lines+media_mod.csv",
    Path(base_path) / "lines.csv"
]

# Verify files exist
for file in input_files:
    if file.exists():
        print(f"✓ Found: {file.name}")
    else:
        print(f"✗ Missing: {file}")

In [ ]:
# Read all input files and check their structure
dataframes = []
for i, file_path in enumerate(input_files):
    print(f"\nReading {file_path.name}...")
    df = pd.read_csv(file_path)
    print(f"  - Shape: {df.shape}")
    print(f"  - Columns: {list(df.columns)}")
    
    # Add source identifier to track which file each line came from
    df['source_file'] = file_path.name
    df['source_index'] = i
    
    dataframes.append(df)
    
    # Show sample (first 3 rows)
    display(df.head(3))

In [ ]:
# Combine all dataframes - this merges all lines from all files
combined_df = pd.concat(dataframes, ignore_index=True)

# Detect the frame column name (could be 'frame', 'Frame', or similar)
frame_col = None
for col in combined_df.columns:
    if col.lower() == 'frame':
        frame_col = col
        break

# Sort by frame first (if frame column exists), then by source to keep organized
if frame_col:
    combined_df = combined_df.sort_values([frame_col, 'source_index']).reset_index(drop=True)
    print(f"Sorting by '{frame_col}' column")
else:
    print("Warning: No 'frame' column found. Data will be sorted by source only.")
    combined_df = combined_df.sort_values('source_index').reset_index(drop=True)

print(f"Combined dataframe shape: {combined_df.shape}")
print(f"Total rows: {len(combined_df)}")

In [ ]:
# Show summary statistics
if frame_col:
    frames = combined_df[frame_col].unique()
    lines_per_frame = combined_df.groupby(frame_col).size()
    
    print("\nSummary Statistics:")
    print(f"  - Total frames: {len(frames)}")
    print(f"  - Lines per frame (mean): {lines_per_frame.mean():.1f}")
    print(f"  - Lines per frame (min): {lines_per_frame.min()}")
    print(f"  - Lines per frame (max): {lines_per_frame.max()}")
    print(f"  - Lines per frame (std): {lines_per_frame.std():.2f}")
    
    # Show distribution
    print("\nLines per frame distribution:")
    print(lines_per_frame.value_counts().sort_index())
    
    # Show breakdown by source file for first frame
    first_frame = frames[0]
    print(f"\nExample - Frame {first_frame} breakdown by source:")
    frame_data = combined_df[combined_df[frame_col] == first_frame]
    print(frame_data.groupby('source_file').size())
    
    # Preview combined data - show one complete frame
    print("\nSample: All lines from first frame:")
    display(combined_df[combined_df[frame_col] == combined_df[frame_col].min()])
else:
    print("No frame column found - showing first 20 rows:")
    display(combined_df.head(20))

In [ ]:
# Define output file
output_file = Path(base_path) / "combined_traces.csv"

# Create a clean version without the tracking columns
output_df = combined_df.drop(columns=['source_file', 'source_index'])

# Save combined data
output_df.to_csv(output_file, index=False)
print(f"Saved combined traces to: {output_file}")
print(f"File size: {output_file.stat().st_size / 1024 / 1024:.2f} MB")
print(f"\nOutput contains {len(output_df)} rows")
if frame_col:
    print(f"Across {output_df[frame_col].nunique()} frames")
else:
    print("All lines from all 4 source files are combined!")